1. Создать новый Jupyter Notebook.
2. В первой ячейке задать номер варианта:

```python
VARIANT = 1  # от 1 до 15
```

3. Запустить ячейку с данными, которая приведена ниже.
4. Реализовать две задачи:
   - `solve_phonebook(contacts)`;
   - `solve_dispatcher(operations, variant)`.
5. Напечатать два ответа:

```python
print(ANSWER1)
print(ANSWER2)
```

6. Создать текстовый отчёт в формате Markdown с LaTeX-формулой ответа.

---

# 2. Ячейка с общими данными для всех вариантов

Эту ячейку студент запускает без изменений, кроме номера варианта.

```python
VARIANT = 1  # от 1 до 15


def make_contacts(v):
    names = [
        "Анна", "Борис", "Вера", "Глеб", "Дарья",
        "Егор", "Зоя", "Иван", "Кира", "Лев"
    ]

    def pretty(digits):
        return f"+{digits[0]} {digits[1:4]} {digits[4:7]}-{digits[7:9]}-{digits[9:11]}"

    def name(shift):
        return names[(v + shift) % len(names)]

    base_digits = f"7900111{20 + v:02d}{30 + v:02d}"
    base8 = "8" + base_digits[1:]
    base10 = base_digits[1:]

    tail_a = "79000009876"
    tail_b = "79000006798"

    contacts = [
        f"{name(0)}; {pretty(base_digits)}",
        f"{name(1)}; {pretty(base8)}",
        f"{name(2)}; {base10[:3]} {base10[3:6]}-{base10[6:8]}-{base10[8:10]}",
        f"{name(0)}; {pretty(base_digits)}",
        f"{name(3)}; {pretty(tail_a)}",
        f"{name(4)}; {pretty(tail_b)}",
        f"{name(5)}; +7 900 555-{v:02d}-{(v + 3) % 100:02d}",
        f"{name(6)}; +7 900 555-{v:02d}-{(v + 3) % 100:02d}",
        f"{name(7).upper()}; +7 900 777-{v:02d}-{(v + 1) % 100:02d}",
        "Строка без точки с запятой",
        "Ещё одна битая строка",
        f"{name(5)}; 12345",
        f"; +7 900 111-11-11",
        f"{name(6)}; 8-999-000-11-22",
    ]

    if v % 2 == 0:
        contacts.append("Дополнительная битая строка для чётного варианта")

    if v % 3 == 0:
        contacts.append(f"{name(8)}; 12-34")

    if v % 5 == 0:
        contacts.append(f"{name(9)}; 8-999-000-11-22")

    return contacts


def make_operations(v):
    payload = {
        'a': v,
        'b': v * 3,
        's': f'var{v}',
        v: 'numeric-key'
    }

    operations = [
        ('add', (v, 5), {}),
        ('add', (v,), {}),
        ('mul', (2, v), {}),
        ('div', (10, 0), {}),
        ('div', (v + 10, 2), {}),
        ('mod', (v, 0), {}),
        ('mod', (v + 7, 4), {}),
        ('int', (str(v + 1),), {}),
        ('int', ('bad',), {}),
        ('get', (payload, 'a'), {}),
        ('get', (payload, 'missing'), {}),
        ('get', (payload, v), {}),
        ('get', (payload, 's', 'default'), {}),
        ('concat', ('x', v, 'y'), {'sep': '-'}),
        ('concat', ('x', v), {'sep': v}),
        ('power', (2, 3), {}),
    ]

    if v % 2 == 0:
        operations.append(('mod', (1, 0), {}))

    if v % 3 == 0:
        operations.append(('unknown_extra', (), {}))

    if v % 5 == 0:
        operations.append(('concat', ('v', v), {'sep': '='}))

    return operations, payload


CONTACTS = make_contacts(VARIANT)
OPERATIONS, PAYLOAD = make_operations(VARIANT)
```

После выполнения этой ячейки доступны:

- `CONTACTS` — список строк для первой задачи;
- `OPERATIONS` — список операций для второй задачи;
- `PAYLOAD` — словарь, который используется внутри некоторых операций.

---

# 3. Задача 1. Обработка телефонного справочника

## Формулировка

Нужно реализовать функцию:

```python
def solve_phonebook(contacts):
    ...
    return answer1
```

Аргумент `contacts` — список строк вида:

```text
"Имя; телефон"
```

Но строки могут быть битыми или некорректными.

---

## Правила обработки контактов

Для каждой строки:

1. Если в строке нет символа `;`, строка считается **битой**.
   - Увеличить счётчик `broken`.
   - Дальше строку не обрабатывать.

2. Если символ `;` есть, разрезать строку один раз:

```python
name_raw, phone_raw = line.split(';', 1)
```

3. Нормализовать имя:

```python
name = name_raw.strip().lower()
```

Если имя пустое, контакт считается **некорректным**.

4. Нормализовать телефон.

Оставить только цифры. Например:

```python
digits = ''.join(ch for ch in phone_raw if ch.isdigit())
```

Далее:

- если длина `10` и номер начинается с `'9'`, добавить спереди `'7'`;
- если длина `11` и номер начинается с `'8'`, заменить первую цифру на `'7'`;
- телефон считается корректным, если после всех исправлений:
  - длина равна `11`;
  - начинается с `'7'`.

Если телефон некорректный, контакт считается **некорректным**.

5. Если имя и телефон корректны, добавить кортеж `(name, phone)` в список корректных пар.

---

## Удаление дубликатов

Нужно:

- собрать все корректные пары в список;
- удалить одинаковые пары `(name, phone)`;
- получить список уникальных пар;
- отсортировать его.

Например:

```python
unique_pairs = list(set(raw_pairs))
unique_pairs.sort()
```

Это должно быть сделано с использованием списка, множества и кортежей.

---

## Построение словарей

### Словарь телефонов

Построить словарь:

```python
by_phone = {
    phone: set_of_names,
    ...
}
```

где `phone` — нормализованный телефон, а `set_of_names` — множество имён, относящихся к этому телефону.

Использовать метод словаря `setdefault`, например:

```python
by_phone.setdefault(phone, set()).add(name)
```

### Словарь «хвостов» телефонов

Построить словарь:

```python
tail_groups = {
    frozenset(phone[-4:]): set_of_phones,
    ...
}
```

Ключ — `frozenset` из последних четырёх цифр телефона.

Значение — множество уникальных телефонов, у которых такой набор цифр в последних четырёх знаках.

---

## Что нужно посчитать

1. `shared_phones` — количество телефонов, у которых больше одного имени:

```python
len(by_phone[phone]) >= 2
```

2. `tail_collisions` — количество ключей-`frozenset`, которым соответствует больше одного уникального телефона:

```python
len(tail_groups[key]) >= 2
```

3. `unique_names` — количество уникальных имён среди уникальных пар.

4. `bad` — сумма:

```python
broken + invalid
```

5. `phone_checksum` — контрольная сумма всех цифр всех уникальных телефонов:

```python
phone_checksum = sum(int(ch) for phone in by_phone for ch in phone) % 1000
```

---

## Ответ задачи 1

Ответ должен быть одним целым числом:

```python
answer1 = (
    shared_phones * 1_000_000
    + tail_collisions * 100_000
    + unique_names * 10_000
    + bad * 1_000
    + phone_checksum
)
```

То есть:

```python
ANSWER1 = solve_phonebook(CONTACTS)
```

---

# 4. Задача 2. Диспетчер функций, исключения и pickle

## Формулировка

Нужно реализовать функцию:

```python
def solve_dispatcher(operations, variant):
    ...
    return answer2
```

Каждый элемент `operations` — это кортеж:

```python
(operation_name, args_tuple, kwargs_dict)
```

Например:

```python
('add', (5, 3), {})
('concat', ('x', 10), {'sep': '-'})
```

Вызывать функцию нужно так:

```python
result = func(*args, **kwargs)
```

---

## Какие функции нужно реализовать

Нужно реализовать следующие функции:

```python
def op_add(a, b):
    return a + b


def op_mul(a, b):
    return a * b


def op_div(a, b):
    return a // b


def op_mod(a, b):
    return a % b


def op_int(value):
    return int(value)


def op_concat(*parts, sep=''):
    if not isinstance(sep, str):
        raise TypeError('sep must be str')
    return sep.join(str(part) for part in parts)


def op_get(data, key):
    return data[key]
```

Важно:

- `op_get` должна обращаться через `data[key]`, чтобы отсутствующий ключ вызывал `KeyError`;
- `op_concat` должна принимать неименованные аргументы через `*parts` и именованный параметр `sep`;
- если `sep` не является строкой, должна возникать `TypeError`.

---

## Диспетчер функций

Нужно создать словарь функций:

```python
dispatch = {
    'add': op_add,
    'mul': op_mul,
    'div': op_div,
    'mod': op_mod,
    'int': op_int,
    'concat': op_concat,
    'get': op_get,
}
```

Это и есть хранение функций в словаре с последующим вызовом.

---

## Обработка операций

Для каждой операции:

1. Если имени операции нет в `dispatch`, увеличить счётчик `unknown`.

2. Если операция есть, вызвать её через:

```python
func(*args, **kwargs)
```

3. Обработать исключения:

- `ZeroDivisionError` → счётчик `zero`;
- `KeyError` → счётчик `key`;
- `TypeError` или `ValueError` → счётчик `type`;
- любое другое исключение → счётчик `error`.

4. Если функция отработала успешно:

- увеличить счётчик `ok`;
- добавить результат в список `results`.

---

## Подсчёты после выполнения операций

После обработки всех операций нужно посчитать:

```python
numeric_sum = sum(
    x for x in results
    if isinstance(x, int) and not isinstance(x, bool)
)
```

```python
text_len = sum(
    len(x) for x in results
    if isinstance(x, str)
)
```

---

## Запись и чтение через pickle

Нужно создать словарь:

```python
data = {
    'ok': ok,
    'unknown': unknown,
    'zero': zero,
    'key': key,
    'type': type_,
    'error': error,
    'numeric_sum': numeric_sum,
    'text_len': text_len,
}
```

Затем сохранить его в бинарный файл:

```python
variant_{variant}.pickle
```

Например, для 5 варианта:

```text
variant_5.pickle
```

Потом прочитать файл обратно через `pickle.load`.

---

## Ответ задачи 2

После чтения данных из файла нужно вычислить:

```python
errors_total = unknown + zero + key + type_ + error
```

Ответ:

```python
answer2 = numeric_sum * 1000 + text_len * 10 + errors_total
```

То есть:

```python
ANSWER2 = solve_dispatcher(OPERATIONS, VARIANT)
```

---

# 5. Итоговый вывод и Markdown/LaTeX-отчёт

После реализации обеих задач студент должен выполнить ячейку примерно такого вида:

```python
import pickle

ANSWER1 = solve_phonebook(CONTACTS)
ANSWER2 = solve_dispatcher(OPERATIONS, VARIANT)

print("ANSWER1:", ANSWER1)
print("ANSWER2:", ANSWER2)

FINAL_ANSWER = f"{ANSWER1}|{ANSWER2}"
print("FINAL:", FINAL_ANSWER)

bs = chr(92)
report = f"""# Вариант {VARIANT}

ANSWER1: {ANSWER1}

ANSWER2: {ANSWER2}

${bs}boxed{{{ANSWER1}|{ANSWER2}}}$
"""

with open(f"variant_{VARIANT}_report.md", "w", encoding="utf-8") as f:
    f.write(report)
```

После этого создаются:

- `variant_{VARIANT}.pickle` — бинарный файл из задачи 2;
- `variant_{VARIANT}_report.md` — текстовый Markdown-файл с ответом, оформленным как LaTeX-формула.

Пример содержимого отчёта:

```markdown
# Вариант 1

ANSWER1: 2184234

ANSWER2: 16168

$\boxed{2184234|16168}$
```

---

# 6. Ограничения и рекомендации

Чтобы проверка действительно соответствовала пройденному материалу, можно ввести ограничения:

1. Не использовать:
   ```python
   collections.Counter
   collections.defaultdict
   ```
   Словари нужно реализовывать через обычные `dict` и метод `setdefault`.

2. В первой задаче обязательно использовать:
   - списки;
   - кортежи;
   - множества;
   - `frozenset`;
   - словари;
   - строковые методы.

3. Во второй задаче обязательно:
   - хранить функции в словаре;
   - вызывать функции через `*args` и `**kwargs`;
   - обрабатывать исключения;
   - использовать `pickle`;
   - создавать бинарный файл;
   - создавать текстовый отчёт.

4. Решение должно быть алгоритмически разумным:
   - не делать лишние вложенные циклы там, где можно использовать множество или словарь;
   - понимать, что поиск в словаре/множестве в среднем работает быстро.

---

In [ ]:
f

In [ ]:
kiselevpg@gmail.com